# Traducción automática de aspectos (Paso 1 del enfoque híbrido)

Traduce cada valor distinto de `aspecto` en `gold.nlp_aspectos_resenas` a español, y los guarda en una tabla de mapeo aparte (`gold.aspecto_traducciones`) para revisar después qué se agrupa bien solo y qué necesita una decisión de negocio (ver notas del final).

**No hace falta el entorno de PyABSA para esto** -- es una librería ligera de traducción, funciona en tu Python normal.

**Diseño incremental y resistente a cortes**, igual que en las tareas 2.1 y 2.2: procesa los aspectos por orden de frecuencia (los más repetidos primero), guarda cada uno según lo traduce, y se puede interrumpir y retomar sin perder lo ya hecho.

## Paso 1 — Instalar dependencias

In [4]:
%pip install -q deep-translator sqlalchemy psycopg2-binary python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## Paso 2 — Conectar

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()
engine = create_engine(os.environ['AZURE_DB_URL'], pool_pre_ping=True, pool_recycle=280)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)

Conectado. PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 3 — Obtener los aspectos distintos aún no traducidos, por orden de frecuencia

Se crea primero la tabla de mapeo (si no existe), y se traen los aspectos que falten, empezando por los más repetidos -- así, si el proceso se corta o decides no terminarlo entero, ya tienes cubierto lo que más impacto tiene.

In [2]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS gold.aspecto_traducciones (
            aspecto_original text PRIMARY KEY,
            aspecto_traducido text,
            frecuencia integer
        )
    '''))
print('Tabla gold.aspecto_traducciones lista (creada si no existia).')

df_pendientes = pd.read_sql('''
    SELECT a.aspecto, COUNT(*) AS frecuencia
    FROM gold.nlp_aspectos_resenas a
    WHERE a.aspecto IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 FROM gold.aspecto_traducciones t WHERE t.aspecto_original = a.aspecto
      )
    GROUP BY a.aspecto
    ORDER BY frecuencia DESC
''', engine)

print('Aspectos distintos pendientes de traducir:', len(df_pendientes))
print(df_pendientes.head(10))

Tabla gold.aspecto_traducciones lista (creada si no existia).
Aspectos distintos pendientes de traducir: 3311
         aspecto  frecuencia
0          owner         452
1            vue         416
2       terrasse         352
3        piscine         279
4         shower         183
5  swimming pool         178
6        zwembad         153
7          küche         122
8        ligging         117
9     apartments         117


## Paso 3b — Limpiar traducciones ya contaminadas (páginas de error guardadas como si fueran válidas)

**Ejecuta esto una sola vez**, si ya lanzaste el Paso 4 antes de este arreglo. Borra las filas que se guardaron con el texto de una página de error de Google en vez de una traducción real, para que se vuelvan a intentar con la lógica corregida.

In [ ]:
with engine.begin() as conn:
    resultado = conn.execute(text('''
        DELETE FROM gold.aspecto_traducciones
        WHERE LENGTH(aspecto_traducido) > 200
           OR aspecto_traducido ILIKE '%error 500%'
           OR aspecto_traducido ILIKE '%server error%'
           OR aspecto_traducido ILIKE '%that''s an error%'
           OR aspecto_traducido ILIKE '%please try again later%'
    '''))
print('Filas contaminadas borradas:', resultado.rowcount)
print('Vuelve a ejecutar el Paso 3 para actualizar la lista de pendientes antes de seguir con el Paso 4.')

## Paso 4 — Traducir por bloques, guardando según se avanza

Usa el traductor gratuito de Google (vía `deep-translator`) -- no necesita clave de API, pero al ser gratuito puede tener límites de ritmo, por eso lleva reintentos con espera progresiva, igual que hicimos con la API de TripAdvisor.

In [3]:
import time
from deep_translator import GoogleTranslator

traductor = GoogleTranslator(source='auto', target='es')

MARCADORES_ERROR = ['error 500', 'server error', "that's an error", 'please try again later', "that's all we know"]


def es_traduccion_valida(texto):
    if not texto:
        return False
    if len(texto) > 200:  # un aspecto real nunca deberia ser tan largo
        return False
    texto_lower = texto.lower()
    return not any(marcador in texto_lower for marcador in MARCADORES_ERROR)


def traducir_con_reintentos(texto, intentos_max=4):
    espera = 2
    for intento in range(intentos_max):
        try:
            resultado = traductor.translate(texto)
            if es_traduccion_valida(resultado):
                return resultado
            print('    respuesta invalida (parece pagina de error) para "' + str(texto) + '" -- esperando', espera, 'segundos (intento', intento + 1, 'de', intentos_max, ')')
        except Exception as error:
            print('    error traduciendo "' + str(texto) + '" -- esperando', espera, 'segundos (intento', intento + 1, 'de', intentos_max, '):', error)
        time.sleep(espera)
        espera = min(espera * 2, 30)
    return None  # si falla del todo, se deja sin traducir para reintentar otro dia

In [4]:
GUARDAR_CADA = 20  # cuantas traducciones acumular antes de escribir en la base de datos

buffer = []
procesados = 0
total = len(df_pendientes)
inicio = time.time()

for _, fila in df_pendientes.iterrows():
    original = fila['aspecto']
    traducido = traducir_con_reintentos(original)
    traducido_normalizado = traducido.strip().lower() if traducido else None

    buffer.append({
        'aspecto_original': original,
        'aspecto_traducido': traducido_normalizado,
        'frecuencia': int(fila['frecuencia']),
    })
    procesados += 1
    time.sleep(0.3)  # pausa entre peticiones para no saturar el traductor gratuito

    if len(buffer) >= GUARDAR_CADA or procesados == total:
        df_buffer = pd.DataFrame(buffer)
        df_buffer.to_sql('aspecto_traducciones', engine, schema='gold', if_exists='append', index=False)
        buffer = []

        transcurrido = time.time() - inicio
        velocidad = procesados / transcurrido if transcurrido > 0 else 0
        restantes = total - procesados
        eta_min = (restantes / velocidad / 60) if velocidad > 0 else 0
        print(f'Guardados {procesados}/{total} -- {velocidad:.1f} aspectos/seg -- estimado restante: {eta_min:.0f} min')

print()
print('Completado. Total traducido en esta ejecucion:', procesados)

Guardados 20/3311 -- 1.9 aspectos/seg -- estimado restante: 29 min
Guardados 40/3311 -- 1.8 aspectos/seg -- estimado restante: 30 min
Guardados 60/3311 -- 1.7 aspectos/seg -- estimado restante: 32 min
Guardados 80/3311 -- 1.6 aspectos/seg -- estimado restante: 33 min
Guardados 100/3311 -- 1.6 aspectos/seg -- estimado restante: 33 min
Guardados 120/3311 -- 1.7 aspectos/seg -- estimado restante: 32 min
Guardados 140/3311 -- 1.7 aspectos/seg -- estimado restante: 31 min
Guardados 160/3311 -- 1.7 aspectos/seg -- estimado restante: 31 min
    respuesta invalida (parece pagina de error) para "decoration" -- esperando 2 segundos (intento 1 de 4 )
    respuesta invalida (parece pagina de error) para "miejsc parkingowych" -- esperando 2 segundos (intento 1 de 4 )
    respuesta invalida (parece pagina de error) para "тераса" -- esperando 2 segundos (intento 1 de 4 )
    respuesta invalida (parece pagina de error) para "gestore" -- esperando 2 segundos (intento 1 de 4 )
Guardados 180/3311 -- 1.5 

## Paso 5 — Revisar qué se ha agrupado, y cuánto queda como caso único

Esto es lo que hay que mirar con ojo humano antes de usarlo -- las traducciones agrupan bien lo que es la misma palabra en otro idioma, pero conceptos relacionados con palabras distintas (ej. "apartment"/"casa"/"place") seguirán apareciendo como grupos separados. Esos son los que quedarían para una decisión de negocio.

In [5]:
with engine.connect() as conn:
    grupos_grandes = pd.read_sql('''
        SELECT aspecto_traducido, 
               COUNT(*) AS num_variantes_originales,
               SUM(frecuencia) AS total_menciones,
               STRING_AGG(aspecto_original, ', ' ORDER BY frecuencia DESC) AS variantes
        FROM gold.aspecto_traducciones
        WHERE aspecto_traducido IS NOT NULL
        GROUP BY aspecto_traducido
        ORDER BY total_menciones DESC
        LIMIT 40
    ''', conn)

pd.set_option('display.max_colwidth', 200)
display(grupos_grandes)

,aspecto_traducido,num_variantes_originales,total_menciones,variantes
0,ubicación,82,8391,"location, ubicación, lokalizacja, locatie, localización, ligging, lokalita, lokacija, sijainti, localização, beliggenhet, lokalizacji, elhelyezkedés, месторасположение, läge, местоположение, staðs..."
1,departamento,57,7791,"apartment, apartamento, appartamento, apartament, wohnung, квартира, mieszkanie, flat, apartman, apartmán, lägenhet, apartamentul, leilighet, lakás, apartement, íbúð, квартире, apartma, byt, apart..."
2,piscina,45,3103,"pool, piscina, piscine, swimming pool, zwembad, basen, бассейн, bazén, басейн, medence, basenu, baseinas, swimmingpool, uima - allas, basenem, basenie, басейну, bassein, бассейном, poolanlage, бас..."
3,anfitrión,35,2612,"host, anfitrión, gastgeber, hôte, gospodarz, gospodarzem, szállásadó, anfitrion, gastheer, anfitrião, házigazda, hostitel, värd, gazdă, värden, gospodarza, gazdei, gestgjafi, domacin, gostitelj, m..."
4,vista,28,2435,"view, vue, vista, widok, вид, kilátás, widokiem, výhled, utsikt, видом, udsigt, výhľad, zicht, vederea, pogled, priveliste, útsýni, priveliște, näköala, výhľadom, veduta, nayra, vyhled, widoku, si..."
5,lugar,43,2156,"place, lugar, miejsce, endroit, posto, lieu, vieta, plek, spot, место, helyen, místo, sted, miejscu, місце, placé, месте, luogo, miejscówka, plaats, místě, място, locale, paikka, sítio, hely, koht..."
6,alojamiento,94,2103,"alojamiento, logement, unterkunft, accommodation, rooms, ubytování, szállás, alloggio, accomodation, помешкання, accommodatie, chambres, ubytovanie, camere, accomodatie, жилье, stanze, pokoje, kam..."
7,casa,15,1684,"casa, house, maison, haus, huis, дом, dům, hús, будинок, hauses, 房 子, домик, 房, kuca, 房 屋"
8,personal,32,1681,"staff, personal, personnel, personale, персонал, personeel, personál, személyzet, personalas, персонала, personalul, personnels, personelem, staf, osoblje, персоналом, starfsfólk, staffs, personàl..."
9,vistas,9,1617,"vistas, views, widoki, vistes, výhledy, widokami, uitzichten, razgledi, vues"


## Notas

- **Esto es el Paso 1 del enfoque híbrido** -- une automáticamente lo que es la misma palabra en distinto idioma (ej. `location`/`ubicación`/`posizione` deberían acabar juntas bajo `aspecto_traducido = 'ubicación'`). No une conceptos relacionados con palabras distintas (`apartment`/`casa`/`place`) -- eso queda para el Paso 2, unas pocas decisiones manuales de negocio sobre el resultado del Paso 5.
- **Revisión humana necesaria antes de usar esto en ningún informe**: la traducción automática de palabras sueltas sin contexto de frase a veces se equivoca. Mira la tabla del Paso 5 con atención.
- El traductor gratuito de Google puede tener límites de uso -- si ves muchos errores seguidos en el Paso 4, para la ejecución y reinténtalo más tarde; gracias al diseño incremental, no se pierde lo ya traducido.
- No he podido probar esto contra el traductor real ni contra el volumen completo de ~11.000 aspectos -- si algo en el formato de la librería no coincide, pégame el error exacto.